<h1>Chapter 9 - Multimodal Understanding</h1>
<i>Analyzing Images with your Agent.</i>


<a href="https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents/blob/main/chapter09/chapter09.ipynb)

---

This notebook is for Chapter 9 of [An Illustrated Guide to AI Agents](https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ) by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
## We use gemma4:e4b-it-qat since there's an issue with vision in Ollama most likely fixed with https://github.com/ollama/ollama/pull/16879
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b-it-qat &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use. 

> Note that we use `gemma4:e4b-it-qat` since there's an issue with vision in Ollama which is most likely resolved with [this PR](https://github.com/ollama/ollama/pull/16879), when it's fixed, we will update this notebook accordingly.

In [ ]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 4 E4B (with native thinking and tool calling)
llm = LLM(model="gemma4:e4b-it-qat", think=True)

If you want to use another LLM, here are a couple of options (both locally and on the cloud) that you can try:

In [ ]:
# # llama.cpp
# llm = LLM(model="gemma-4-e4B-it-Q4_K_M", base_url="http://127.0.0.1:8080")

# # LM Studio
# llm = LLM(model="gemma-4-e4b-it", base_url="http://127.0.0.1:1234/v1")

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-4-31b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

## 2 - Adding Multimodal Understanding

The model that we have been using `Gemma 4 E4B` is a multimodal model and is capable of processing images, audio, and video alongside text. If we were to use the raw chat template, you would get something like this:


The raw chat template creates a special token for a patch of pixels:

<pre style="background:#1e1e1e; color:#d4d4d4; padding:16px; border-radius:8px; overflow-x:auto; font-size:13px; line-height:1.6;">
<span style="color:#b392f0;">&lt;bos&gt;</span><span style="color:#f97583;">&lt;|turn&gt;system</span>
You are a helpful assistant.<span style="color:#f97583;">&lt;turn|&gt;</span>
<span style="color:#f97583;">&lt;|turn&gt;user</span>
<span style="color:#79b8ff;">&lt;|image&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;image|&gt;</span>Which animal is on the cover of 'Hands-On Large Language Models'?<span style="color:#f97583;">&lt;turn|&gt;</span>
<span style="color:#f97583;">&lt;|turn&gt;model</span>
</pre>

However, in `Ollama`, this automatically parsed using the `messages` that we have been leveraging, namely like so:

```json
[
    {
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": {
                    "url": "https://www.oreilly.com/covers/urn:orm:book:9798341662681/300w/"
                },
            },
            {"type": "text", "text": "What’s in this image?"},
        ],
    }
]
```

Note that some LLM inference engines do not support using a URL, and you would have to use the base64 encoded image instead:

```json
[
    {
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": {"url": "data:image/png;base64,..."},
            },
            {"type": "text", "text": "What’s in this image?"},
        ],
    }
]
```


Note how the `content` key now knows how to separate dictionaries, one containing `text` and the other `image_url`. This allows `Ollama` to process the image and feed it to the LLM. This means that we will have to adjust how the message structure is being used, which requires two changes.

* `memory.py` - Add a parameter to add the `"image_url"`
* `agent.py` - Only add the `"image_url"` to memory when the user provides an image url

Let's explore these changes, starting with `memory.py`:

In [ ]:
from illustrated_agents.chapters.ch4 import Memory

class MultimodalMemory(Memory):
    """Simple memory module to store conversation history."""

    def add(
        self,
        role: str,
        content: str,
        tool_call: dict = None,
        image_data: str = None,
    ):
        """Add a message to memory."""
        # Image
        if image_data:
            is_url = image_data.startswith(("http://", "https://"))
            url = image_data if is_url else f"data:image/png;base64,{image_data}"
            content = [
                {"type": "image_url", "image_url": {"url": url}},
                {"type": "text", "text": content},
            ]

        # Main message
        message = {"role": role, "content": content}

        # Tool call
        if tool_call:
            message["tool_calls"] = [tool_call]

        # Append message to memory
        self.messages.append(message)

Note how straightforward these changes are, we merely need to update `.add` with the `image_url`. The changes are minimal:

In [ ]:
from illustrated_agents.chapters.ch9 import memory_diff; memory_diff

## 3 - Updating `TinyAgent`

The changes to the `TinyAgent` are also minimal and instead of showing you the full code for the `TinyAgent`, we are going to create a new class that inherits all capabilities but simply adds the `image_url=image_url` when the user's task is first added to `Memory`:

In [ ]:
from illustrated_agents.chapters import ch6


class TinyAgent(ch6.TinyAgent):
    def run(self, task: str, image_data: str = None) -> str:
        """Run the agent on a task."""
        self.memory.add("user", task, image_data=image_data)
        self.trajectory.initialize(task)

        # *Autonomy* loop
        for step in range(self.planner.max_steps):
            result = self._step()
            if result is not None:
                return result

        return "Max steps reached without completion."

Only two lines of code need to be changed in order to add this multimodal capabilities to your `TinyAgent`:

In [ ]:
from illustrated_agents.chapters.ch9 import tinyagents_diff; tinyagents_diff

## 3 - Running the Multimodal Agent

Now that you have the necessary components, you can create your `TinyAgent` and give it an image to analyze.

In [ ]:
from illustrated_agents.chapters.ch5 import NativeTools
from illustrated_agents.chapters.ch6 import NativeReAct

# Multimodal Agent
agent = TinyAgent(
    llm=llm, 
    tools=NativeTools(), 
    memory=MultimodalMemory(),
    planner=NativeReAct()
)

Next up, we will download the cover of "An Illustrated Guide to AI Agents" and encode it to base64 so that the openai endpoint can properly process the image:

In [ ]:
import base64
import httpx

# Download and encode the image
image_url = "https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png"
image_data = base64.b64encode(httpx.get(image_url).content).decode("utf-8")

Finally, we simply ask the Agent which animal is on the cover. If it can correctly view the image then it should get the answer correct:

In [ ]:
agent.run("Which animal is on the cover of 'Hands-On Large Language Models'?", image_data=image_data)

The answer is correct, let's see how the model got to that conclusion:

In [ ]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent.trajectory)

Although we're not tracking the image input, we can still see it in the Agent's memory:

In [ ]:
from rich.pretty import pprint

pprint(agent.memory.get_messages(), max_string=150)

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

In this chapter, we covered how to give your `TinyAgent` the ability to process images alongside text. This was only possible because the underlying model, Gemma 4, had native multimodal capabilities!

In [ ]:
from illustrated_agents.chapters.ch9 import what_we_built; what_we_built

# What's Next

In the next chapter, we will explore how to go from your general purpose `TinyAgent` to a `CodingAgent`! It is going to be an interesting exploration as we cover what it means to give an Agent more access to your environment. Security is therefore key ;)


